In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, TimestampType

In [0]:
dbutils.widgets.text("bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")


BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")


BRONZE_TABLE = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_historical_voivodeships_weather"
SILVER_PARENT_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_historical_voivodeship_weather"
SILVER_TARGET_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_historical_weather_metrics_for_clustering"

In [0]:
df_bronze = spark.read.table(BRONZE_TABLE)
df_silver_parent = (
    df_bronze
    .withColumn("lat", col("lat").cast(DoubleType()))
    .withColumn("lon", col("lon").cast(DoubleType()))
    .withColumn("time", col("time").cast(TimestampType()))
    .withColumn("temperature_2m", col("temperature_2m").cast(DoubleType()))
    .withColumn("precipitation", col("precipitation").cast(DoubleType()))
    .withColumn("wind_speed_10m", col("wind_speed_10m").cast(DoubleType()))
    .withColumn("soil_temp", col("soil_temp").cast(DoubleType()))
    .withColumn("humidity_2m", col("humidity_2m").cast(DoubleType()))
    .withColumn("dew_point_2m", col("dew_point_2m").cast(DoubleType()))
)

In [0]:
(df_silver_parent.write
 .mode("overwrite")
 .partitionBy("voivodeship")
 .format("delta")
 .saveAsTable(SILVER_PARENT_TABLE))

In [0]:

df_silver_weather_metrics = df_silver_parent.select(
    "voivodeship",
    "lat",
    "lon",
    "time",
    "precipitation",
    "soil_temp"
)

(df_silver_weather_metrics.write
 .mode("overwrite")
 .partitionBy("voivodeship")
 .format("delta")
 .saveAsTable(SILVER_TARGET_TABLE))

In [0]:
# df_1 = spark.read.table(SILVER_METRICS_TABLE)
# display(df_1.limit(30))

In [0]:
# df_2 = spark.read.table(SILVER_PARENT_TABLE)
# display(df_2.limit(30))